In [ ]:
!pip uninstall diffusers -y -q
!pip install git+https://github.com/huggingface/diffusers.git -q
!pip install peft accelerate transformers -q

import os
os.kill(os.getpid(), 9)

In [ ]:
# ติดตั้ง
!pip install diffusers transformers accelerate -q
!pip install bitsandbytes -q
!pip install peft -q

import torch
import os
from google.colab import drive

drive.mount('/content/drive')

BASE       = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER = f'{BASE}/images_human_fixed'
LORA_FOLDER  = f'{BASE}/models_lora'
os.makedirs(LORA_FOLDER, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f'VRAM: {vram/1e9:.1f} GB')
    if vram < 12e9:
        print('VRAM น้อยกว่า 12GB')
        print('แนะนำใช้ LoRA แทน full fine-tune')
    else:
        print('VRAM พอสำหรับ fine-tune')

In [ ]:
import torch
import os
import json
import pandas as pd
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

BASE         = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER = f'{BASE}/images_human_fixed'
LORA_FOLDER  = f'{BASE}/models_lora'
TRAIN_FOLDER = '/content/cdt_train_images'
OUTPUT_DIR   = '/content/lora_output'

os.makedirs(LORA_FOLDER,  exist_ok=True)
os.makedirs(TRAIN_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Diffusers: {__import__("diffusers").__version__}')

n_imgs = len([
    f for f in os.listdir(TRAIN_FOLDER)
    if f.endswith('.jpg')
]) if os.path.exists(TRAIN_FOLDER) else 0

print(f'Training images: {n_imgs} รูป')

if n_imgs < 10:
    print('สร้าง training data ใหม่')
else:
    print('Training data')

In [ ]:
import pandas as pd
from PIL import Image
import shutil

train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)

good_df = train_df[
    (train_df['domain_A'] == 2) &
    (train_df['domain_D'] == 2) &
    (train_df['domain_E'] == 2)
].sample(
    min(200, len(train_df)),
    random_state=42
)

print(f'รูปสำหรับ fine-tune: {len(good_df)} รูป')

TRAIN_FOLDER = '/content/cdt_train_images'
os.makedirs(TRAIN_FOLDER, exist_ok=True)

captions = []
for _, row in good_df.iterrows():
    src  = f'{HUMAN_FOLDER}/{row["filename"]}'
    dst  = f'{TRAIN_FOLDER}/{row["filename"]}'

    try:
        img = Image.open(src).convert('RGB')
        img = img.resize((512, 512))
        img.save(dst)

        caption = (
            "a CDT clock drawing test image, "
            "hand drawn by elderly patient, "
            "pencil on white paper, "
            "handwritten numbers and clock hands"
        )
        captions.append({
            'file_name': row['filename'],
            'text'     : caption
        })
    except:
        continue

import json
with open(
    f'{TRAIN_FOLDER}/metadata.jsonl', 'w'
) as f:
    for cap in captions:
        f.write(json.dumps(cap) + '\n')

print(f'เสร็จ: {len(captions)} รูป')

In [ ]:
!accelerate config default

FINETUNE_SCRIPT = """
accelerate launch --mixed_precision=fp16 \\
  train_text_to_image_lora.py \\
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \\
  --train_data_dir="{train_folder}" \\
  --caption_column="text" \\
  --resolution=512 \\
  --random_flip \\
  --train_batch_size=1 \\
  --gradient_accumulation_steps=4 \\
  --num_train_epochs=50 \\
  --learning_rate=1e-4 \\
  --lr_scheduler="cosine" \\
  --lr_warmup_steps=0 \\
  --output_dir="{output_dir}" \\
  --checkpointing_steps=500 \\
  --seed=42
""".format(
    train_folder=TRAIN_FOLDER,
    output_dir=LORA_FOLDER
)

!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py

print('Training command:')
print(FINETUNE_SCRIPT)

In [ ]:
import subprocess
import time
print("Start")

start = time.time()

result = subprocess.run(
    FINETUNE_SCRIPT,
    shell=True,
    capture_output=False
)

elapsed = time.time() - start
print(f'\nFine-tune Done')
print(f'ใช้เวลา: {elapsed/60:.1f} นาที')

!cp -r /content/lora_output/* {LORA_FOLDER}/
print(f'Save finish')

In [ ]:
import json
import os

import pandas as pd
from PIL import Image

TRAIN_FOLDER = '/content/cdt_train_images'
os.makedirs(TRAIN_FOLDER, exist_ok=True)

train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)
good_df = train_df[
    (train_df['domain_A'] == 2) &
    (train_df['domain_D'] == 2) &
    (train_df['domain_E'] == 2)
].sample(min(100, len(train_df)), random_state=42)

metadata_path = f'{TRAIN_FOLDER}/metadata.jsonl'
saved = 0

with open(metadata_path, 'w') as f:
    for _, row in good_df.iterrows():
        src = f'{HUMAN_FOLDER}/{row["filename"]}'
        dst = f'{TRAIN_FOLDER}/{row["filename"]}'

        try:
            img = Image.open(src).convert('RGB')
            img = img.resize((512, 512))
            img.save(dst)

            entry = {
                'file_name': row['filename'],
                'text': (
                    "a CDT clock drawing test, "
                    "hand drawn by elderly patient, "
                    "pencil on white paper, sks style"
                )
            }
            f.write(json.dumps(entry) + '\n')
            saved += 1
        except:
            continue

print(f'สร้าง metadata {saved} รูป')
with open(metadata_path) as f:
    print(f.readline())

In [ ]:
import diffusers
print(f'Diffusers version: {diffusers.__version__}')
# ควรได้ >= 0.39.0.dev0

In [ ]:
!pip install torchao --upgrade -q

In [ ]:
!sed -i 's/from peft.tuners.lora.torchao/# from peft.tuners.lora.torchao/' \
    /usr/local/lib/python3.12/dist-packages/peft/tuners/lora/model.py

In [ ]:
import subprocess
import time
import os
import shutil

!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py

assert os.path.exists('train_text_to_image_lora.py'), "ดาวน์โหลด script ไม่ได้"
print("Start")

TRAIN_FOLDER = "/content/cdt_train_images"

OUTPUT_DIR = "/content/lora_output"
LORA_FOLDER = "/content/lora_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LORA_FOLDER, exist_ok=True)

!accelerate launch \
    --mixed_precision=no \
    train_text_to_image_lora.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --train_data_dir="{TRAIN_FOLDER}" \
    --caption_column="text" \
    --image_column="image" \
    --resolution=512 \
    --random_flip \
    --train_batch_size=1 \
    --gradient_accumulation_steps=4 \
    --num_train_epochs=30 \
    --learning_rate=1e-4 \
    --lr_scheduler="cosine" \
    --lr_warmup_steps=0 \
    --output_dir="{OUTPUT_DIR}" \
    --seed=42

if os.path.exists(OUTPUT_DIR):
    files = os.listdir(OUTPUT_DIR)
    print(f'\nFiles in output: {files}')

    if files:
        shutil.copytree(
            OUTPUT_DIR,
            LORA_FOLDER,
            dirs_exist_ok=True
        )
        print('บันทึก LoRA weights แล้ว')
    else:
        print('ไม่มีไฟล์ใน output folder')
else:
    print('ไม่พบ output folder')

In [ ]:
from diffusers import (
    StableDiffusionInpaintPipeline,
    StableDiffusionPipeline
)

pipe_ft = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16,
    safety_checker=None,
)

pipe_ft.load_lora_weights(LORA_FOLDER)
pipe_ft = pipe_ft.to(device)
pipe_ft.enable_attention_slicing()

In [ ]:
import matplotlib.pyplot as plt

prompts = {
    'class2_perfect': (
        "a CDT clock drawing test, "
        "hand drawn by elderly patient, "
        "pencil on white paper, "
        "complete clock with all 12 numbers "
        "and two hands pointing to 11 and 2"
    ),
    'class0_no_hands': (
        "a CDT clock drawing test, "
        "hand drawn by elderly patient, "
        "pencil on white paper, "
        "clock face with numbers but NO hands, "
        "missing clock hands"
    ),
    'class1_one_hand': (
        "a CDT clock drawing test, "
        "hand drawn by elderly patient, "
        "pencil on white paper, "
        "clock with only ONE hand, "
        "incomplete clock drawing"
    ),
    'class0_missing_nums': (
        "a CDT clock drawing test, "
        "hand drawn by elderly patient, "
        "pencil on white paper, "
        "clock with missing numbers, "
        "only some numbers visible"
    ),
}

fig, axes = plt.subplots(
    2, 4, figsize=(20, 10)
)

for idx, (name, prompt) in enumerate(
    prompts.items()
):
    for sample in range(2):
        ax = axes[sample][idx]

        with torch.no_grad():
            out = pipe_ft(
                prompt=prompt,
                negative_prompt=(
                    "colorful, digital clock, "
                    "perfect clean drawing, "
                    "young person, printed"
                ),
                num_inference_steps=30,
                guidance_scale=7.5,
                width=512, height=512,
            )

        img = out.images[0]
        ax.imshow(img)
        ax.set_title(
            f'{name}\nsample {sample+1}',
            fontsize=8
        )
        ax.axis('off')

plt.suptitle(
    'Fine-tuned SD — CDT Style Generation',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{BASE}/results/lora_generation_test.png',
    dpi=120, bbox_inches='tight'
)
plt.show()

In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageDraw
import random

def find_clock_center_v2(img_pil):
    img_cv = cv2.cvtColor(
        np.array(img_pil), cv2.COLOR_RGB2GRAY
    )
    w, h = img_pil.size
    _, thresh = cv2.threshold(
        img_cv, 200, 255, cv2.THRESH_BINARY_INV
    )
    contours, _ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return w//2, h//2, min(w,h)//3
    largest  = max(contours, key=cv2.contourArea)
    x, y, bw, bh = cv2.boundingRect(largest)
    cx = x + bw//2
    cy = y + bh//2
    r  = max(bw, bh)//2
    return cx, cy, r

def create_hands_mask_v2(img_pil):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    cx, cy, r_clock = find_clock_center_v2(img_pil)
    r_mask = int(r_clock * 0.4)
    draw.ellipse(
        [cx-r_mask, cy-r_mask,
         cx+r_mask, cy+r_mask],
        fill=255
    )
    return mask

def create_numbers_mask_v4(img_pil, n_missing=4):
    w, h = img_pil.size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    cx, cy, r_clock = find_clock_center_v2(img_pil)
    num_angles = {
        12: -90,  1: -60,  2: -30,
         3:   0,  4:  30,  5:  60,
         6:  90,  7: 120,  8: 150,
         9: 180, 10: 210, 11: 240,
    }
    numbers_to_mask = random.sample(
        list(num_angles.keys()), n_missing
    )
    size = max(15, int(r_clock * 0.15))
    for num in numbers_to_mask:
        rad = np.radians(num_angles[num])
        x   = int(cx + r_clock * 0.72 * np.cos(rad))
        y   = int(cy + r_clock * 0.72 * np.sin(rad))
        x   = max(size, min(w-size, x))
        y   = max(size, min(h-size, y))
        draw.ellipse(
            [x-size, y-size, x+size, y+size],
            fill=255
        )
    return mask, numbers_to_mask

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import numpy as np
import cv2

pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    'runwayml/stable-diffusion-inpainting',
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe_inpaint.load_lora_weights(LORA_FOLDER)
pipe_inpaint = pipe_inpaint.to(device)
pipe_inpaint.enable_attention_slicing()

good_imgs = good_df.sample(3, random_state=42)

fig, axes = plt.subplots(
    3, 5, figsize=(25, 15)
)

for row_idx, (_, row) in enumerate(
    good_imgs.iterrows()
):
    src = f'{HUMAN_FOLDER}/{row["filename"]}'
    img = Image.open(src).convert('RGB').resize(
        (512, 512)
    )

    cx, cy, r = find_clock_center_v2(img)

    hands_mask = create_hands_mask_v2(img)

    num_mask, masked_nums = \
        create_numbers_mask_v4(img, n_missing=4)

    axes[row_idx][0].imshow(img)
    axes[row_idx][0].set_title(
        f'Original\n{row["filename"][:12]}',
        fontsize=8
    )
    axes[row_idx][0].axis('off')


    axes[row_idx][1].imshow(
        hands_mask, cmap='gray'
    )
    axes[row_idx][1].set_title(
        'Hands Mask', fontsize=8
    )
    axes[row_idx][1].axis('off')

    # Generate class 0 (no hands)
    out_c0 = pipe_inpaint(
        prompt=(
            "CDT clock drawing, elderly patient, "
            "pencil drawing, NO clock hands, "
            "blank center, white paper"
        ),
        negative_prompt=(
            "clock hands, arrows, lines, "
            "colorful, digital, printed"
        ),
        image=img,
        mask_image=hands_mask,
        num_inference_steps=30,
        guidance_scale=7.5,
        strength=0.85,
    )
    axes[row_idx][2].imshow(out_c0.images[0])
    axes[row_idx][2].set_title(
        'Class 0\n(no hands)', fontsize=8
    )
    axes[row_idx][2].axis('off')

    # Generate class 1 (one hand)
    out_c1 = pipe_inpaint(
        prompt=(
            "CDT clock drawing, elderly patient, "
            "pencil drawing, ONE clock hand only, "
            "single hand, white paper"
        ),
        negative_prompt=(
            "two hands, multiple hands, "
            "colorful, digital, printed"
        ),
        image=img,
        mask_image=hands_mask,
        num_inference_steps=30,
        guidance_scale=7.5,
        strength=0.85,
    )
    axes[row_idx][3].imshow(out_c1.images[0])
    axes[row_idx][3].set_title(
        'Class 1\n(one hand)', fontsize=8
    )
    axes[row_idx][3].axis('off')

    # Generate class 0 digits
    out_d0 = pipe_inpaint(
        prompt=(
            "CDT clock drawing, elderly patient, "
            "pencil drawing, missing numbers, "
            "blank spaces, white paper"
        ),
        negative_prompt=(
            "complete numbers, all numbers, "
            "colorful, digital, printed"
        ),
        image=img,
        mask_image=num_mask,
        num_inference_steps=30,
        guidance_scale=7.5,
        strength=0.75,
    )
    axes[row_idx][4].imshow(out_d0.images[0])
    axes[row_idx][4].set_title(
        f'Class 0 Digits\n(missing {masked_nums})',
        fontsize=8
    )
    axes[row_idx][4].axis('off')

plt.suptitle(
    'Fine-tuned SD Inpainting — CDT Style\n'
    'Col1=Original | Col2=Mask | '
    'Col3=No Hands | Col4=One Hand | '
    'Col5=Missing Digits',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{BASE}/results/lora_inpainting_test.png',
    dpi=120, bbox_inches='tight'
)
plt.show()

In [ ]:
# รัน cell นี้เฉพาะถ้าผล feasibility ดี

GEN_FOLDER_CLASS0 = f'{BASE}/images_generated/class0'
GEN_FOLDER_CLASS1 = f'{BASE}/images_generated/class1'
os.makedirs(GEN_FOLDER_CLASS0, exist_ok=True)
os.makedirs(GEN_FOLDER_CLASS1, exist_ok=True)

TARGET = 500  # รูปต่อ class
gen_labels = []

for idx, (_, row) in enumerate(
    good_df.iterrows()
):
    if idx >= TARGET:
        break

    src = f'{HUMAN_FOLDER}/{row["filename"]}'
    img = Image.open(src).convert('RGB').resize(
        (512, 512)
    )

    cx, cy, r   = find_clock_center_v2(img)
    hands_mask  = create_hands_mask_v2(img)

    # Generate class 0
    out_c0 = pipe_inpaint(
        prompt=(
            "CDT clock drawing test, "
            "elderly patient handwriting, "
            "pencil, NO hands, blank center"
        ),
        negative_prompt="hands, arrows, lines",
        image=img, mask_image=hands_mask,
        num_inference_steps=25,
        guidance_scale=7.5, strength=0.85,
    )
    fname_c0 = f'gen_c0_{idx:04d}.jpg'
    out_c0.images[0].save(
        f'{GEN_FOLDER_CLASS0}/{fname_c0}'
    )

    # Generate class 1
    out_c1 = pipe_inpaint(
        prompt=(
            "CDT clock drawing test, "
            "elderly patient handwriting, "
            "pencil, ONE hand only"
        ),
        negative_prompt="two hands, multiple",
        image=img, mask_image=hands_mask,
        num_inference_steps=25,
        guidance_scale=7.5, strength=0.85,
    )
    fname_c1 = f'gen_c1_{idx:04d}.jpg'
    out_c1.images[0].save(
        f'{GEN_FOLDER_CLASS1}/{fname_c1}'
    )

    # บันทึก label
    gen_labels.append({
        'filename' : fname_c0,
        'domain_D' : 0,
        'domain_E' : 0,
        'domain_A' : int(row['domain_A']),
        'domain_B' : int(row['domain_B']),
        'domain_C' : int(row['domain_C']),
    })
    gen_labels.append({
        'filename' : fname_c1,
        'domain_D' : 1,
        'domain_E' : 1,
        'domain_A' : int(row['domain_A']),
        'domain_B' : int(row['domain_B']),
        'domain_C' : int(row['domain_C']),
    })

    if idx % 50 == 0:
        print(f'Generated: {idx}/{TARGET}')

pd.DataFrame(gen_labels).to_csv(
    f'{BASE}/labels/generated_labels.csv',
    index=False
)
print(f'\n✅ Generate เสร็จ!')
print(f'  Class 0: {TARGET} รูป')
print(f'  Class 1: {TARGET} รูป')